# E034 — Z-Norm Score Normalization for Speaker Verification

**Motivation:** Score normalization (Z-norm, T-norm) is standard in speaker verification to account for test condition variability. Current system uses raw LLR scores which may have session-dependent biases.

**Hypothesis:** Z-norm (normalizing test score by cohort mean/std) will reduce EER by making scores more comparable across test conditions.

**Approach:**
1. Select cohort: 20-30 non-target speakers
2. For each test sample, compute score against cohort models
3. Z-norm: z = (score_test - mean_cohort) / std_cohort
4. Compare normalized vs raw scores

**Variants:**
- `raw`: E025 baseline (no normalization)
- `znorm`: Z-norm with cohort
- `tnorm`: T-norm (cohort = impostor trials)
- `znorm+tnorm`: Both combined

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
import librosa
from scipy.special import logsumexp
from sklearn.mixture import GaussianMixture
import copy

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

SEED = 67
UBM_COMPONENTS = 32
MAP_R = 16.0
COHORT_SIZE = 20  # number of cohort speakers

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
print(f'{len(manifest)} samples')

E025_REF = {'mean_eer': 1.94, 'std_eer': 1.57}

In [ ]:
def _find_wav(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".wav")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _extract_lpcc(y, sr, order=12, n_cep=13, hop_length=160, win_length=400):
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    cep_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            A_freq = np.fft.rfft(a, n=512)
            log_H = -np.log(np.abs(A_freq) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        cep_frames.append(cep)
    feat = np.array(cep_frames, dtype=np.float32)
    delta = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    feat = np.hstack([feat, delta, delta2])
    feat -= feat.mean(axis=0)
    return feat

def _aug_pitch(y, sr, rng):
    n_steps = float(rng.choice([-2, -1, 1, 2]))
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

def _train_ubm(X):
    return GaussianMixture(n_components=UBM_COMPONENTS, covariance_type="diag",
                           max_iter=200, random_state=SEED).fit(X)

def _map_adapt(ubm, X_target):
    log_resp = ubm._estimate_log_prob(X_target) + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp = np.exp(log_resp)
    n_k = resp.sum(axis=0)
    mu_hat = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha = n_k / (n_k + MAP_R)
    adapted = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted

def _train_lpcc_with_cohort(train_df, cohort_df, data_dir, seed):
    """Train target model + cohort models for Z-norm."""
    rng = np.random.default_rng(seed)
    
    # Extract target features
    X_target = []
    for _, row in train_df[train_df['label'] == 1].iterrows():
        y_wav, sr = librosa.load(_find_wav(row["stem"], data_dir), sr=None, mono=True)
        wavs = [y_wav, _aug_pitch(y_wav, sr, rng)]
        for y_aug in wavs:
            X_target.append(_extract_lpcc(y_aug, sr))
    X_target = np.vstack(X_target)
    
    # Extract non-target features for UBM
    X_nontarget = []
    for _, row in train_df[train_df['label'] == 0].iterrows():
        y_wav, sr = librosa.load(_find_wav(row["stem"], data_dir), sr=None, mono=True)
        X_nontarget.append(_extract_lpcc(y_wav, sr))
    X_nontarget = np.vstack(X_nontarget)
    
    # Train UBM
    ubm = _train_ubm(X_nontarget)
    adapted = _map_adapt(ubm, X_target)
    
    # Train cohort models
    cohort_models = []
    cohort_speakers = cohort_df['identity'].unique()[:COHORT_SIZE]
    
    for speaker_id in cohort_speakers:
        speaker_df = cohort_df[cohort_df['identity'] == speaker_id]
        X_cohort = []
        for _, row in speaker_df.iterrows():
            y_wav, sr = librosa.load(_find_wav(row["stem"], data_dir), sr=None, mono=True)
            X_cohort.append(_extract_lpcc(y_wav, sr))
        X_cohort = np.vstack(X_cohort)
        
        # Adapt UBM to cohort speaker
        cohort_model = _map_adapt(ubm, X_cohort)
        cohort_models.append(cohort_model)
    
    return ubm, adapted, cohort_models

def _score_lpcc(wav_path, adapted, ubm):
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    feat = _extract_lpcc(y, sr)
    return float(adapted.score(feat) - ubm.score(feat))

def _znorm_score(score, cohort_models, feat):
    """Compute Z-norm score."""
    cohort_scores = [model.score(feat) - ubm.score(feat) for model in cohort_models]
    cohort_mean = np.mean(cohort_scores)
    cohort_std = np.std(cohort_scores) + 1e-10
    return (score - cohort_mean) / cohort_std

print('Model functions defined')

## 2. Cross-validation: raw vs Z-norm

In [ ]:
print("Running CV (this may take a while)...")

oof_raw = np.full(len(manifest), np.nan)
oof_znorm = np.full(len(manifest), np.nan)

for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    seed_f = SEED + fold_id
    train_df = manifest.loc[train_idx]
    val_df = manifest.loc[val_idx]
    
    # Use non-target train as cohort
    cohort_df = train_df[train_df['label'] == 0]
    
    print(f"Fold {fold_id}: training target + {COHORT_SIZE} cohort models...")
    ubm, adapted, cohort_models = _train_lpcc_with_cohort(train_df, cohort_df, DATA, seed_f)
    
    for idx, row in val_df.iterrows():
        wav_path = _find_wav(row["stem"], DATA)
        y, sr = librosa.load(wav_path, sr=None, mono=True)
        feat = _extract_lpcc(y, sr)
        
        # Raw score
        raw_score = adapted.score(feat) - ubm.score(feat)
        oof_raw[idx] = raw_score
        
        # Z-norm score
        cohort_scores = [model.score(feat) - ubm.score(feat) for model in cohort_models]
        cohort_mean = np.mean(cohort_scores)
        cohort_std = np.std(cohort_scores) + 1e-10
        oof_znorm[idx] = (raw_score - cohort_mean) / cohort_std

# Evaluate
for name, scores in [('raw', oof_raw), ('znorm', oof_znorm)]:
    eer, eer_thr = compute_eer(scores[y_all == 1], scores[y_all == 0])
    min_dcf, _ = compute_min_dcf(scores[y_all == 1], scores[y_all == 0])
    print(f"{name:8s}: EER={eer*100:5.2f}%  min-DCF={min_dcf:.4f}")

## 3. Conclusion

Z-norm effect: [TBD]
Decision: [Adopt/Reject]